In [ ]:
import transformers
print(transformers.__version__)

4.41.2


In [ ]:

!pip uninstall -y transformers accelerate datasets peft numpy

!pip install \
transformers==4.41.2 \
datasets==2.19.0 \
accelerate==0.30.1 \
numpy==1.26.4 \
scikit-learn pandas pyarrow

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
import requests
import pandas as pd

def load_parquet_from_api(api_url, filename):
    response = requests.get(api_url)
    parquet_url = response.json()[0]

    file_data = requests.get(parquet_url)

    with open(filename, "wb") as f:
        f.write(file_data.content)

    return pd.read_parquet(filename)

train = load_parquet_from_api(
    "https://huggingface.co/api/datasets/ai4bharat/indic_glue/parquet/bbca.hi/train",
    "train.parquet"
)

test = load_parquet_from_api(
    "https://huggingface.co/api/datasets/ai4bharat/indic_glue/parquet/bbca.hi/test",
    "test.parquet"
)

print(train.shape, test.shape)

(3467, 2) (866, 2)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

classes = np.unique(train['label'])
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train['label']
)

class_weights = torch.tensor(weights, dtype=torch.float)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train['label'] = le.fit_transform(train['label'])
test['label'] = le.transform(test['label'])

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train[['text', 'label']])
test_ds = Dataset.from_pandas(test[['text', 'label']])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_)
)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=384
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/3467 [00:00<?, ? examples/s]

Map:   0%|          | 0/866 [00:00<?, ? examples/s]

In [ ]:
train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=6,

    weight_decay=0.01,
    logging_dir="./logs",

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",
    greater_is_better=True,

    report_to="none"
)

In [ ]:
from transformers import Trainer
import torch
import torch.nn as nn

# Custom loss for imbalance
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

import os
os.environ["WANDB_DISABLED"] = "true"
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.488270,0.756351,0.724443
2,1.878700,1.459525,0.719400,0.701950
3,1.366800,1.378608,0.756351,0.740883
4,1.147800,1.384996,0.756351,0.732595
5,1.073600,1.394139,0.763279,0.746958
6,0.987400,1.373769,0.765589,0.751172


TrainOutput(global_step=2604, training_loss=1.2727767399379186, metrics={'train_runtime': 3410.2502, 'train_samples_per_second': 6.1, 'train_steps_per_second': 0.764, 'total_flos': 4105369408011264.0, 'train_loss': 1.2727767399379186, 'epoch': 6.0})

In [ ]:
trainer.save_model("./best_model_tuned")

In [ ]:
tokenizer.save_pretrained("./best_model_tuned")

('./best_model_tuned/tokenizer_config.json',
 './best_model_tuned/special_tokens_map.json',
 './best_model_tuned/sentencepiece.bpe.model',
 './best_model_tuned/added_tokens.json',
 './best_model_tuned/tokenizer.json')

In [ ]:
trainer.save_metrics("eval", trainer.evaluate())

In [ ]:
trainer.evaluate()

{'eval_loss': 1.3737691640853882,
 'eval_accuracy': 0.7655889145496536,
 'eval_f1': 0.7511723407796065,
 'eval_runtime': 31.5823,
 'eval_samples_per_second': 27.42,
 'eval_steps_per_second': 3.451,
 'epoch': 6.0}

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
trainer.evaluate()

{'eval_loss': 1.3423495292663574,
 'eval_accuracy': 0.7586605080831409,
 'eval_f1': 0.7367145236923734,
 'eval_runtime': 20.4611,
 'eval_samples_per_second': 42.324,
 'eval_steps_per_second': 5.327,
 'epoch': 5.0}

In [ ]:
!zip -r best_model_tuned.zip best_model_tuned

  adding: best_model_tuned/ (stored 0%)
  adding: best_model_tuned/tokenizer.json (deflated 76%)
  adding: best_model_tuned/sentencepiece.bpe.model (deflated 49%)
  adding: best_model_tuned/tokenizer_config.json (deflated 77%)
  adding: best_model_tuned/config.json (deflated 60%)
  adding: best_model_tuned/model.safetensors (deflated 29%)
  adding: best_model_tuned/training_args.bin (deflated 53%)
  adding: best_model_tuned/special_tokens_map.json (deflated 52%)


In [ ]:
from google.colab import files
files.download("best_model_tuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>